# Lot A2 — Gold, EXP-A2-020 : entraînement final de la cascade (MARBERTv2)

Protocole : `docs/lot-a2_protocole_gold.md` (§9) et avenant 2 du protocole du modèle tunisien.

- **Entrée unique** : `final_train.tsv` + `final_inputs.json`, produits en local par
  `python -X utf8 -m normalization.gold_kaggle_inputs` (jeu de données Kaggle **privé**).
  Aucun texte du test, d'eval_30 ni de la presse n'est envoyé ici.
- **Configuration figée** (EXP-A2-015) : lr 2e-5, 4 époques, lots de 16, longueur 128,
  échauffement 10 %, AdamW (décroissance 0.01) ; graines 1, 2, 3 ; seuil de la cascade 0.30
  (appliqué en local, pas ici).
- **Sorties** : `/kaggle/working/marbert_final_graine{1,2,3}/` (modèle + tokenizer), leurs archives
  `.zip`, et `run_summary.json` (configuration, effectifs, SHA-256 de chaque fichier de modèle).
  **Aucune prédiction, aucun score sur le test, aucun texte affiché.**
- À lancer **une seule fois** (Accelerator : GPU T4).


In [ ]:
# Le GPU doit apparaître ; sinon : Settings > Accelerator > GPU T4
!nvidia-smi --query-gpu=name,memory.total --format=csv
import glob
print(glob.glob("/kaggle/input/**/final_train.tsv", recursive=True))
print(glob.glob("/kaggle/input/**/final_inputs.json", recursive=True))

In [ ]:
# Toutes déjà installées sur Kaggle (torch, transformers, pandas)
import gc
import glob
import hashlib
import json
import os
import random
import shutil
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import transformers
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

In [ ]:
# --- Configuration FIGÉE (EXP-A2-015, avenant 2) : aucune variable d'environnement ---------
MODEL = "UBC-NLP/MARBERTv2"
SEEDS = [1, 2, 3]
LR = 2e-5
EPOCHS = 4
BATCH = 16
MAX_LEN = 128
WARMUP = 0.1
WEIGHT_DECAY = 0.01
THRESHOLD = 0.30  # seuil de la cascade : consigné ici, appliqué en local (gold_predict_local)
OUT = "/kaggle/working"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONFIG = {"model": MODEL, "lr": LR, "epochs": EPOCHS, "batch": BATCH, "max_len": MAX_LEN,
          "warmup": WARMUP, "weight_decay": WEIGHT_DECAY, "seeds": SEEDS,
          "seuil_cascade": THRESHOLD}

In [ ]:
def find_one(pattern: str) -> str:
    """Un seul fichier attendu dans les jeux de données ajoutés au notebook."""
    paths = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    if len(paths) != 1:
        raise SystemExit(f"{pattern} introuvable ou en double : {len(paths)} fichier(s)")
    return paths[0]


def sha256(path: str) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_loader(tokenizer, texts, labels, seed):
    """Même construction que la validation croisée (EXP-A2-015), sur les 800 textes."""
    enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN, padding=True,
                    return_tensors="pt")
    items = [{"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i],
              "labels": torch.tensor(int(labels[i]))} for i in range(len(texts))]
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(items, batch_size=BATCH, shuffle=True, generator=generator)


def train_full(tokenizer, texts, labels, seed):
    """Affine MARBERTv2 sur les 800 textes ; renvoie le modèle et la perte moyenne par époque."""
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(DEVICE)
    loader = make_loader(tokenizer, texts, labels, seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps = EPOCHS * len(loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    losses = []
    for _ in range(EPOCHS):
        model.train()
        total = 0.0
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(device_type=DEVICE, enabled=DEVICE == "cuda"):
                loss = model(**batch).loss
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total += float(loss)
        losses.append(round(total / len(loader), 4))
    return model, losses


def file_digests(folder: str) -> dict:
    """SHA-256 de chaque fichier du dossier du modèle (vérifiés en local avant prédiction)."""
    return {name: sha256(os.path.join(folder, name)) for name in sorted(os.listdir(folder))
            if os.path.isfile(os.path.join(folder, name))}

In [ ]:
def main() -> None:
    start = time.time()
    if os.path.exists(f"{OUT}/run_summary.json"):
        raise SystemExit("run_summary.json existe déjà : l'entraînement final ne se refait pas")
    train_path, info_path = find_one("final_train.tsv"), find_one("final_inputs.json")
    with open(info_path, encoding="utf-8") as handle:
        expected = json.load(handle)["fichiers"]["entrainement"]
    if sha256(train_path) != expected["sha256"]:
        raise SystemExit("final_train.tsv : empreinte différente de final_inputs.json")
    # quoting=3 (QUOTE_NONE) : un guillemet est un caractère ordinaire (bug corrigé en EXP-A2-015)
    data = pd.read_csv(train_path, sep="\t", quoting=3, dtype={"text": str},
                       keep_default_na=False)
    with open(train_path, encoding="utf-8") as handle:
        lines = sum(1 for _ in handle) - 1
    if not (len(data) == lines == expected["n"] == 800):
        raise SystemExit(f"lecture incorrecte : {len(data)} lues, {lines} lignes,"
                         f" {expected['n']} attendues")
    if not set(data["label"]) <= {0, 1} or not set(data["lex"]) <= {0, 1}:
        raise SystemExit("colonnes label / lex invalides")
    texts, labels = data["text"].to_numpy(), data["label"].to_numpy()
    print(f"{len(data)} textes (tn {int(labels.sum())}, ar {int(len(labels) - labels.sum())}) ;"
          f" {DEVICE}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL)

    models = {}
    for seed in SEEDS:
        model, losses = train_full(tokenizer, texts, labels, seed)
        folder = f"{OUT}/marbert_final_graine{seed}"
        model.save_pretrained(folder, safe_serialization=True)
        tokenizer.save_pretrained(folder)
        models[str(seed)] = {"dossier": os.path.basename(folder), "pertes": losses,
                             "fichiers": file_digests(folder)}
        shutil.make_archive(folder, "zip", folder)
        print(f"graine {seed} : pertes par époque {losses} ; {len(models[str(seed)]['fichiers'])}"
              f" fichiers ({time.time() - start:.0f} s)")
        del model
        gc.collect()
        torch.cuda.empty_cache()

    summary = {"experience": "EXP-A2-020", "config": CONFIG,
               "entrainement": {"n": len(data), "sha256": expected["sha256"]},
               "modeles": models,
               "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "minutes": round((time.time() - start) / 60, 1)}
    with open(f"{OUT}/run_summary.json", "w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=1, ensure_ascii=False)
    print(f"run_summary.json écrit ; {summary['minutes']} min ; versions {summary['versions']}")


main()

## Après l'exécution

Télécharger `run_summary.json` et les trois archives `marbert_final_graine{1,2,3}.zip`, les
décompresser **hors du dépôt**, puis lancer en local (inférence CPU, textes jamais envoyés) :

```
python -X utf8 -m normalization.gold_predict_local --entrees <dossier des entrées> \
    --resume <run_summary.json> --modele-graine1 <dossier> --modele-graine2 <dossier> \
    --modele-graine3 <dossier> --out-dir <hors dépôt>
```

Le script vérifie la configuration et l'empreinte de chaque fichier de modèle avant toute
prédiction. Installer localement la **même version** de `transformers` que celle de
`run_summary.json` (`versions`).